# Demo 10a — Hard and soft assignments

**Class 10 · A1** (slides: `slides/10_clustering.md`)

A hard clustering gives each example one label, $z_{ik} \in \{0, 1\}$; a soft clustering gives a degree,
$u_{ik} \in [0, 1]$ with $\sum_k u_{ik} = 1$. Same data, same number of clusters $k$:

* $k$-means (hard) against a Gaussian mixture (soft) on overlapping blobs.
* NMF of Class 09 read as a soft clustering: normalise its weights.
* A sweep over the overlap, and the failure case: soft weights on data with no clusters.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import polars as pl
import pynnmf
from sklearn.cluster import KMeans
from sklearn.datasets import make_blobs
from sklearn.metrics import adjusted_rand_score as ari
from sklearn.mixture import GaussianMixture

from faa import clustering as cl
from faa import demo, plotlib
from faa.demo import show, slide

demo.setup("10")
BLUE, ORANGE, GREEN, RED, SKY, PURPLE, GREY = (plotlib.BLUE, plotlib.ORANGE, plotlib.GREEN, plotlib.RED, plotlib.SKY, plotlib.PURPLE, plotlib.GREY)
PAL = np.array([[0.0, 0.447, 0.698], [0.902, 0.624, 0.0], [0.0, 0.62, 0.451]])  # Okabe-Ito blue, orange, green as RGB
SEED = 0

## Slide: One data set, two kinds of answer

Three overlapping blobs (std 1.2). $k$-means writes a one-hot row per point; the mixture writes a probability row.
The blend of the three colours by $u_{ik}$ is the soft picture.

In [ ]:
X, y = make_blobs(300, centers=3, cluster_std=1.2, random_state=SEED)
km = KMeans(3, n_init=10, random_state=SEED).fit(X)
gm = GaussianMixture(3, covariance_type="full", n_init=3, random_state=SEED).fit(X)
Z = np.eye(3)[km.labels_]  # hard: z_ik in {0, 1}
U = gm.predict_proba(X)  # soft: u_ik in [0, 1], rows sum to 1
order = np.argsort(km.labels_)  # relabel the mixture components to the k-means ones for the colours
perm = [int(np.argmin(((gm.means_ - c) ** 2).sum(1))) for c in km.cluster_centers_]
U = U[:, perm]

f, ax = plt.subplots(1, 2, figsize=(plotlib.FULL, 1.75), sharex=True, sharey=True)
ax[0].scatter(*X.T, c=PAL[km.labels_], s=7, lw=0)
ax[0].set_title("hard: $z_{ik} \\in \\{0, 1\\}$")
ax[1].scatter(*X.T, c=np.clip(U @ PAL, 0, 1), s=7, lw=0)
ax[1].set_title("soft: $u_{ik} \\in [0, 1]$")
for a in ax:
    a.set_xticks([]), a.set_yticks([])
plotlib.save(f, "a1_hard_soft")

slide("Hard against soft")
agree = (U.argmax(1) == km.labels_).mean()
print(f"argmax of the mixture agrees with k-means on {agree:.3f} of the points")
print(f"ARI: k-means {ari(y, km.labels_):.3f}, mixture argmax {ari(y, U.argmax(1)):.3f}")
print(f"each u_ik row sums to {U.sum(1).min():.3f} .. {U.sum(1).max():.3f}")

## Slide: Three points, three rows

A point at the centre of a cluster, one between two clusters and one far out: the hard row is always one-hot,
the soft row says how sure the model is.

In [ ]:
centre = km.cluster_centers_
pts = {
    "centre of cluster 0": centre[0],
    "between 0 and 1": (centre[0] + centre[1]) / 2,
    "far outside": X.mean(0) + 12,
}
rows = []
for name, p in pts.items():
    u = gm.predict_proba(p[None])[0][perm]
    rows.append((name, int(np.argmin(((centre - p) ** 2).sum(1))), *u.round(3), float(cl.entropy(u[None])[0])))
slide("Three points, three rows")
show(pl.DataFrame(rows, schema=["point", "hard label", "u_0", "u_1", "u_2", "entropy (bits)"], orient="row"))

## Slide: Entropy marks the border

The entropy of a membership row, $-\sum_k u_{ik}\log_2 u_{ik}$, is 0 for a certain point and $\log_2 k$ for a
point equally shared by all clusters. Size and darkness of a point show it.

In [ ]:
ent = cl.entropy(U)
f = plt.figure(figsize=(plotlib.FULL, 1.75))
gs = f.add_gridspec(1, 4, width_ratios=[1.1, 0.05, 0.6, 1.1], wspace=0.15)
ax = [f.add_subplot(gs[0]), f.add_subplot(gs[3])]
sc = ax[0].scatter(*X.T, c=ent, s=4 + 18 * ent, cmap="viridis", lw=0)
ax[0].set_xticks([]), ax[0].set_yticks([])
f.colorbar(sc, cax=f.add_subplot(gs[1]))
ax[0].set_title("colour and size: entropy (bits)", fontsize=7)
ax[1].hist(ent, bins=24, color=BLUE)
ax[1].set_xlabel("entropy (bits)"), ax[1].set_ylabel("points")
plotlib.save(f, "a1_entropy")
slide("Entropy marks the border")
print(f"entropy: median {np.median(ent):.3f}, 90th percentile {np.quantile(ent, 0.9):.3f}, maximum {np.log2(3):.3f}")
print(f"points with entropy above 0.5 bit: {(ent > 0.5).sum()} of {len(ent)}")

## Slide: NMF is a soft clustering

Class 09 factored $X^\top \approx WH$ with $W, H \geq 0$. Read the columns of $W$ as prototypes (archetypes) and
the columns of $H$ as weights; normalise each column of $H$ to sum to 1 and it is a membership row $u_i$.
NMF needs $X \geq 0$: split each centred feature into its positive and negative part.

In [ ]:
Xc = X - X.mean(0)
Xp = np.c_[np.maximum(Xc, 0), np.maximum(-Xc, 0)]  # n x 4, all >= 0
rec, W, H, err = pynnmf.nmf_hals(Xp.T, 3, n=300, seed=SEED, init="random")  # W: 4 x 3, H: 3 x n
Unmf = cl.nmf_membership(W, H)
arche = (W[:2] - W[2:]).T  # undo the split: each archetype back in the original 2 features
corr = np.corrcoef(Unmf.T, U.T)[:3, 3:]  # 3 x 3 between NMF and mixture memberships
best = corr.argmax(1)
slide("NMF is a soft clustering")
print(f"shapes: W {W.shape} (archetypes), H {H.shape} (weights); relative error {np.linalg.norm(Xp.T - W @ H) / np.linalg.norm(Xp):.3f}")
print(f"NMF argmax ARI {ari(y, Unmf.argmax(1)):.3f}, mixture {ari(y, U.argmax(1)):.3f}, k-means {ari(y, km.labels_):.3f}")
print(f"membership correlation with the mixture, best match per NMF part: {np.round([corr[j, best[j]] for j in range(3)], 3)}")

f, ax = plt.subplots(1, 2, figsize=(plotlib.FULL, 1.75), sharex=True, sharey=True)
ax[0].scatter(*Xc.T, c=np.clip(Unmf @ PAL, 0, 1), s=7, lw=0)
for j in range(3):
    ax[0].annotate("", xy=arche[j] * 1.0, xytext=(0, 0), arrowprops={"arrowstyle": "->", "color": "k", "lw": 1})
ax[0].set_title("NMF memberships, archetypes $w_j$")
ax[1].scatter(*Xc.T, c=np.clip(U @ PAL, 0, 1), s=7, lw=0)
ax[1].set_title("mixture posterior")
for a in ax:
    a.set_xticks([]), a.set_yticks([])
plotlib.save(f, "a1_nmf_soft")

## Sweep: how much overlap makes the rows soft?

Same three centres, growing cluster std. Hard labels stay crisp at every overlap; the mean top membership of the
soft model falls, and that is the information the hard label hides.

In [ ]:
rows = []
for std in (0.4, 0.8, 1.2, 1.6, 2.0, 2.5):
    Xs, ys = make_blobs(300, centers=[(-4, 0), (4, 0), (0, 6)], cluster_std=std, random_state=SEED)
    g = GaussianMixture(3, n_init=3, random_state=SEED).fit(Xs)
    Us = g.predict_proba(Xs)
    k = KMeans(3, n_init=10, random_state=SEED).fit(Xs)
    rows.append((std, ari(ys, k.labels_), ari(ys, Us.argmax(1)), Us.max(1).mean(), float((cl.entropy(Us) > 0.5).mean())))
sweep = pl.DataFrame(rows, schema=["std", "ARI k-means", "ARI mixture", "mean top u", "share entropy > 0.5"], orient="row")
slide("Overlap sweep")
show(sweep)
f, ax = plt.subplots(figsize=(plotlib.COL, 1.5))
ax.plot(sweep["std"], sweep["mean top u"], "o-", color=BLUE, label="mean top $u_{ik}$")
ax.plot(sweep["std"], sweep["ARI mixture"], "s-", color=ORANGE, label="ARI")
ax.set_xlabel("cluster std (overlap grows)"), ax.legend()
plotlib.save(f, "a1_overlap")

## Failure: soft weights on data without clusters

Uniform noise has no groups. A hard clustering still returns $k$ crisp labels. A mixture also fits, and its
memberships are not flat: soft does not mean "honest" about the absence of structure; a high entropy needs the
data to be ambiguous between real clusters.

In [ ]:
rng = np.random.default_rng(SEED)
Xu = rng.uniform(-1, 1, (300, 2))
gu = GaussianMixture(3, n_init=3, random_state=SEED).fit(Xu)
Uu = gu.predict_proba(Xu)
slide("No clusters, still crisp rows")
print(f"uniform data: mean top membership {Uu.max(1).mean():.3f}, share of points with entropy > 0.5: {(cl.entropy(Uu) > 0.5).mean():.3f}")
print(f"three real blobs (std 1.2): mean top membership {U.max(1).mean():.3f}, share entropy > 0.5: {(ent > 0.5).mean():.3f}")
print("silhouette is the check for structure: see demo 10f")

## Slide: Three points, three rows (figure)

The three rows of the table as stacked bars: the hard row is one block, the soft row splits it.

In [ ]:
rows3 = np.array([gm.predict_proba(p[None])[0][perm] for p in pts.values()])
f, ax = plt.subplots(1, 2, figsize=(plotlib.FULL, 1.3), sharey=True)
for a, M, title in ((ax[0], np.eye(3)[[km.predict(p[None])[0] for p in pts.values()]], "hard $z_{i\\cdot}$"), (ax[1], rows3, "soft $u_{i\\cdot}$")):
    left = np.zeros(3)
    for j in range(3):
        a.barh(range(3), M[:, j], left=left, color=PAL[j], height=0.6, label=f"cluster {j}")
        left += M[:, j]
    a.set_yticks(range(3), list(pts)), a.set_title(title), a.set_xlim(0, 1), a.grid(False)
ax[1].legend(ncol=3, loc="lower center", bbox_to_anchor=(-0.1, -0.5), fontsize=6)
plotlib.save(f, "a1_rows")

## Slide: Soft does not mean honest (figure)

The uniform points, size and colour by the top membership of a three-component mixture: crisp rows, no clusters.

In [ ]:
f, ax = plt.subplots(1, 2, figsize=(plotlib.FULL, 1.55), sharex=True, sharey=True)
ax[0].scatter(*Xu.T, c=np.clip(Uu @ PAL, 0, 1), s=3 + 14 * Uu.max(1), lw=0)
ax[0].set_title("uniform data, three components")
ax[1].scatter(*X.T, c=np.clip(U @ PAL, 0, 1), s=3 + 14 * U.max(1), lw=0)
ax[1].set_title("three real blobs")
for a in ax:
    a.set_xticks([]), a.set_yticks([])
plotlib.save(f, "a1_uniform")